In [2]:
import cv2
import numpy as np


def preprocess(image):
    image = cv2.resize(image, (224, 224))
    blur = cv2.GaussianBlur(image, (5, 5), 0)
    hsv = cv2.cvtColor(blur, cv2.COLOR_BGR2HSV)
    return image, blur, hsv


def detect_road(hsv):
    lower = np.array([0, 0, 50])
    upper = np.array([180, 80, 220])

    road_mask = cv2.inRange(hsv, lower, upper)

    height, width = road_mask.shape
    polygon = np.array([[
        (20, height),
        (85, 135),
        (140, 135),
        (204, height)
    ]], dtype=np.int32)

    roi_mask = np.zeros_like(road_mask)
    cv2.fillPoly(roi_mask, polygon, 255)

    road_mask = cv2.bitwise_and(road_mask, roi_mask)

    kernel = np.ones((5, 5), np.uint8)
    road_mask = cv2.morphologyEx(
        road_mask, cv2.MORPH_CLOSE, kernel, iterations=2
    )
    road_mask = cv2.morphologyEx(
        road_mask, cv2.MORPH_OPEN, kernel, iterations=1
    )

    return road_mask


def detect_lanes(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)

    height, width = edges.shape

    polygon = np.array([[
        (20, height),
        (85, 135),
        (140, 135),
        (204, height)
    ]], dtype=np.int32)

    mask = np.zeros_like(edges)
    cv2.fillPoly(mask, polygon, 255)

    roi = cv2.bitwise_and(edges, mask)

    lines = cv2.HoughLinesP(
        roi,
        1,
        np.pi / 180,
        threshold=30,
        minLineLength=20,
        maxLineGap=10
    )

    left = []
    right = []

    if lines is None:
        return None, None

    lines = lines.reshape(-1, 4)

    for x1, y1, x2, y2 in lines:

        if x1 == x2:
            continue

        slope = (y2 - y1) / (x2 - x1)
        length = np.hypot(x2 - x1, y2 - y1)

        if abs(slope) < 0.5 or length < 25:
            continue

        if max(y1, y2) < 160:
            continue

        bottom_x = x1 if y1 > y2 else x2

        if slope < 0 and bottom_x < width / 2:
            left.append((x1, y1, x2, y2, length))

        elif slope > 0 and bottom_x > width / 2:
            right.append((x1, y1, x2, y2, length))

    return make_lane(left, height), make_lane(right, height)


def make_lane(lines, height):
    if not lines:
        return None

    slopes = []
    intercepts = []
    weights = []

    for x1, y1, x2, y2, length in lines:
        slope = (y2 - y1) / (x2 - x1)
        intercept = y1 - slope * x1

        slopes.append(slope)
        intercepts.append(intercept)
        weights.append(length)

    slope = np.average(slopes, weights=weights)
    intercept = np.average(intercepts, weights=weights)

    if abs(slope) < 1e-6:
        return None

    y1 = height
    y2 = 135

    x1 = int((y1 - intercept) / slope)
    x2 = int((y2 - intercept) / slope)

    return x1, y1, x2, y2


def draw_result(image, road_mask, left_lane, right_lane):
    output = image.copy()

    blue = np.zeros_like(output)
    blue[:] = (255, 0, 0)

    road_area = road_mask > 0

    output[road_area] = cv2.addWeighted(
        output[road_area], 0.65,
        blue[road_area], 0.35, 0
    )

    for lane in [left_lane, right_lane]:
        if lane is not None:
            x1, y1, x2, y2 = lane
            cv2.line(
                output,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                4
            )

    return output


def process_frame(frame):
    resized, blur, hsv = preprocess(frame)

    road_mask = detect_road(hsv)
    left_lane, right_lane = detect_lanes(blur)

    output = draw_result(
        resized,
        road_mask,
        left_lane,
        right_lane
    )

    return output


def main():
    input_video = "input-road.mp4"
    output_video = "road_mask_lane_output.mp4"

    cap = cv2.VideoCapture(input_video)

    if not cap.isOpened():
        print("Video not found!")
        return

    input_fps = cap.get(cv2.CAP_PROP_FPS)

    if input_fps <= 0:
        input_fps = 30

    target_fps = 3
    frame_step = max(1, round(input_fps / target_fps))

    writer = cv2.VideoWriter(
        output_video,
        cv2.VideoWriter_fourcc(*"mp4v"),
        target_fps,
        (224, 224)
    )

    frame_count = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        if frame_count % frame_step == 0:
            output = process_frame(frame)
            writer.write(output)

        frame_count += 1

    cap.release()
    writer.release()

    print("Output saved:", output_video)


if __name__ == "__main__":
    main()

Output saved: road_mask_lane_output.mp4
